# Data and tests Initation

In [1]:
!pip install datasets
!pip install apache_beam
!pip install sentence_transformers
# !pip install -U datasets fsspec huggingface_hub ## Might be necessary

  Using cached pyarrow-18.1.0-cp313-cp313-win_amd64.whl.metadata (3.4 kB)
Using cached pyarrow-18.1.0-cp313-cp313-win_amd64.whl (25.1 MB)
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 23.0.0
    Uninstalling pyarrow-23.0.0:
      Successfully uninstalled pyarrow-23.0.0


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.4.1 requires pyarrow>=21.0.0, but you have pyarrow 18.1.0 which is incompatible.


  Using cached sentence_transformers-5.2.0-py3-none-any.whl.metadata (16 kB)
  Using cached transformers-4.57.6-py3-none-any.whl.metadata (43 kB)
  Using cached torch-2.10.0-cp313-cp313-win_amd64.whl.metadata (31 kB)
  Using cached scikit_learn-1.8.0-cp313-cp313-win_amd64.whl.metadata (11 kB)
  Using cached scipy-1.17.0-cp313-cp313-win_amd64.whl.metadata (60 kB)
  Using cached huggingface_hub-0.36.0-py3-none-any.whl.metadata (14 kB)
  Using cached regex-2026.1.15-cp313-cp313-win_amd64.whl.metadata (41 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
  Using cached safetensors-0.7.0-cp38-abi3-win_amd64.whl.metadata (4.2 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached markupsafe-3.0.3-cp313-cp313-win_amd64.whl.metadata (2.8 kB)
  Using ca

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.4.1 requires pyarrow>=21.0.0, but you have pyarrow 18.1.0 which is incompatible.


In [2]:
from datasets import load_dataset

# loading wikipedia using HuggingFace
wikipedia = load_dataset('wikimedia/wikipedia', '20231101.en')

# Searching for the New Zealand's wikipedia passage
for passage in wikipedia['train']:
  if passage['title'] == 'New Zealand':
    new_zealand_passage = passage['text']
    break

# Breaking the passage into a corpus of sentences.
corpus = new_zealand_passage.split('.')
print(corpus)

c:\Users\Inbal\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Inbal\miniconda3\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Inbal\.cache\huggingface\hub\datasets--wikimedia--wikipedia. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this

['New Zealand ( ) is an island country in the southwestern Pacific Ocean', ' It consists of two main landmasses—the North Island () and the South Island ()—and over 700 smaller islands', ' It is the sixth-largest island country by area and lies east of Australia across the Tasman Sea and south of the islands of New Caledonia, Fiji, and Tonga', " The country's varied topography and sharp mountain peaks, including the Southern Alps, owe much to tectonic uplift and volcanic eruptions", " New Zealand's capital city is Wellington, and its most populous city is Auckland", '\n\nThe islands of New Zealand were the last large habitable land to be settled by humans', ' Between about 1280 and 1350, Polynesians began to settle in the islands and then developed a distinctive Māori culture', ' In 1642, the Dutch explorer Abel Tasman became the first European to sight and record New Zealand', ' In 1840, representatives of the United Kingdom and Māori chiefs signed the Treaty of Waitangi, which in its

In [3]:
test_querys = ['What is the capital of New Zealand?',
               'What currency is used in New Zealand?',
               'Should I visit New Zealand if I am interested in sightseeing?']

# Sparse Search - TF-IDF
### Implement

In [4]:
def computeTF(token: str, doc: str):
  words = doc.lower().split()
  c = words.count(token.lower())
  d = len(words)
  return c / d

In [5]:
def computeDF(token: str, corpus: list):
  return sum(1 for doc in corpus if token.lower() in doc.lower().split())

In [6]:
import numpy as np
def computeIDF(token: str, corpus: list):
  c = len(corpus)
  df =  computeDF(token,corpus)
  return  np.log(c/(df+1))

In [7]:
def computeTFIDF(token: str, doc: str, corpus: list):
  return computeIDF(token,corpus)*computeTF(token, doc)

In [8]:
import heapq

def sparse_search(query: str, corpus: list, num_results_to_return: int=3):
  query_tokens = query.lower().split()
  heap = []
  query_idfs = {token: computeIDF(token, corpus) for token in query_tokens}
  
  for doc in corpus:
    score = sum(computeTF(token, doc)*query_idfs[token] for token in query_tokens)
    if len(heap)<num_results_to_return:
      heapq.heappush(heap, (score, doc))
    else:
      if score > heap[0][0]:
        heapq.heappushpop(heap, (score, doc))
          
  top_k_sorted = sorted(heap, key=lambda x: x[0], reverse=True)
  return [item[1] for item in top_k_sorted]

### Test your implementation
Use the Preloaded corpus, and the test querys

In [ ]:
"""
def test_sparse_search(queries, corpus):
    print(f"--- Testing Sparse Search (TF-IDF) ---")
    
    for i, query in enumerate(queries):
        print(f"\nQuery {i+1}: {query}")
        
        results = sparse_search(query, corpus, num_results_to_return=3)
        
        if not results:
            print("  No results found.")
        else:
            for rank, result in enumerate(results, 1):
                print(f"  Result {rank}: {repr(result.strip())}")

test_sparse_search(test_querys, corpus)

"""

--- Testing Sparse Search (TF-IDF) ---

Query 1: What is the capital of New Zealand?
  Result 1: "New Zealand's capital city is Wellington, and its most populous city is Auckland"
  Result 2: 'New Zealand'
  Result 3: 'The South Island is the largest landmass of New Zealand'

Query 2: What currency is used in New Zealand?
  Result 1: 'The word today is increasingly used to refer to all non-Polynesian New Zealanders'
  Result 2: 'New Zealand'
  Result 3: 'While the demonym for a New Zealand citizen is New Zealander, the informal "Kiwi" is commonly used both internationally and by locals'

Query 3: Should I visit New Zealand if I am interested in sightseeing?
  Result 1: 'New Zealand'
  Result 2: 'If no majority is formed, a minority government can be formed if support from other parties during confidence and supply votes is assured'
  Result 3: 'Public radio was introduced in New Zealand in 1922'


# Dense Search

### Implement

In [9]:
from sentence_transformers import SentenceTransformer
import numpy as np
model_name = 'sentence-transformers/all-mpnet-base-v2'

In [12]:
# Ingestion - run this only once, not per query

model = SentenceTransformer(model_name)
embedded_corpus = model.encode(corpus)
print(embedded_corpus.shape)

c:\Users\Inbal\miniconda3\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Inbal\.cache\huggingface\hub\models--sentence-transformers--all-mpnet-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


(491, 768)


In [ ]:
def dense_search(query: str, embedded_corpus: np.ndarray, model: SentenceTransformer, num_results_to_return: int=3):
  query_embedding = model.encode(query)
  scores = np.dot(embedded_corpus, query_embedding)
  top_indices = np.argsort(scores)[-num_results_to_return:][::-1]
  return [corpus[i] for i in top_indices]
  

### Test your implementation
Use the Embedded corpus, and the test querys